[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/rendering/examples/electromagnetism/piezoelectric/piezoelectric.ipynb)

# Piezoelectric Bimorph

A piezoelectric bimorph bends when a voltage makes one bonded layer expand and the other contract. The same strip works as a sensor: bending it moves charge between its electrodes, or develops a voltage when the circuit is open. This notebook solves both experiments with a continuum material law and a finite-element mesh.

The algebra is planar spacetime, `x+y+t-`. Displacements and polarization are intrinsic spatial directions; the electric field is a plane through time. The piezoelectric map couples strain to polarization. Its adjugate supplies the converse mechanical response, so actuation and sensing share one constitutive coupling.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga@rendering"
%matplotlib inline

In [ ]:
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import STA2D as ga
from numga.sparse import SparseExtensor

from examples.electromagnetism.piezoelectric.numerical import solve_coupled
from examples.electromagnetism.piezoelectric import render
from examples.electromagnetism.piezoelectric.core import Mesh, boundary_maps, electrode_potential, strip, tip_load
from examples.util.animation import save_figure, show_animation

# The spacetime of the strip's plane, x+y+t-: two positive spatial directions, one negative time direction.
ctx = NumpyContext(ga)
mv = ctx.multivector
Scalar = ga.gatype.scalar()
# Spatial planes read positions, displacements and polarization, their intrinsic complements.
Plane = ga.gatype.from_blades("x y")
Point = Plane.dual().output_gatype
# Electric fields are extrinsic planes through time; charge is an intrinsic amount.
Field = ga.gatype.from_blades("xt yt")
Amount = ga.gatype.antiscalar()
Strain = ga.gatype((Plane, Point))
Elastic = ga.gatype((Scalar, Point, Plane, Point, Plane))
Polarization = ga.gatype((Point, Point, Plane))
Permittivity = ga.gatype((Point, Field))
time = -mv.t.dual()
axis = mv.x.dual()
pole = mv.y.dual()

## Laminated strip

The two PZT layers are bonded at their middle surface, with their poling directions pointing apart. The left edge is clamped; metal electrodes cover the top and bottom. The arrows show the material's poling, the dark line its bonded interface. The model uses small strains and plane stress through the width.

The strip's mesh comes from the core, its topology and its vertices. The picture shows its clamped end: the triangles, the electrodes along the top and bottom, the interface between the layers, and the clamped nodes.

In [ ]:
length_unit = 1e-3
stress_unit = 1e10
permittivity_unit = 1e-8
field_unit = np.sqrt(stress_unit / permittivity_unit)
voltage_unit = length_unit * field_unit
force_unit = stress_unit * length_unit**2
charge_unit = force_unit * length_unit / voltage_unit

length, thickness, width = 30, 1, 5
n_length, n_thickness = 180, 12
applied_voltage = 100 / voltage_unit
applied_force = 0.06 / force_unit
displacement_scale, strain_scale = 100, 2e-5
n_frames, duration_ms = 90, 40

In [ ]:
mesh, vertices, poling = strip(length, thickness, n_length, n_thickness)
save_figure(render.draw_mesh(mesh, vertices), "piezoelectric_mesh");


## Triangles

Each triangle's corners are read off the vertices by its face, and each corner's opposite edge runs between the other two. Twice a triangle's signed area is its two edges joined, read against time: intrinsic, since a winding reverses in a mirror. A corner's shape function rises from zero along the opposite edge to one at the corner, so its gradient is the plane through that edge and time, `edges & time`, over twice the signed area. The integrals over the strip weight each triangle by its area's size, an ordinary number.

In [ ]:
corners = vertices.batch()[mesh.faces]                                            # [faces, 3] Point-
# Each corner's opposite edge, from the next corner to the one after it.
edges = corners[:, [1, 2, 0]] - corners[:, [2, 0, 1]]                             # [faces, 3] Point-
double_area = (edges[:, 0] & edges[:, 1]) & time                                  # [faces] Scalar-
gradients = (edges & time) / double_area[:, None]                                 # [faces, 3] Plane+
areas = double_area.norm() / 2                                                    # [faces] Scalar+


def strain(displacement: Point, gradients: Plane, faces: np.ndarray) -> Strain:
    """The symmetric displacement gradient on each triangle, as a map from directions to stretch readings."""
    movement = displacement.batch()[..., faces]
    gradient = (movement.dual_inverse() * (gradients & Point)).sum(axis=-1)
    return (gradient + gradient.adjugate()) / 2


zero_motion = vertices * 0
zero_potential = electrode_potential(mesh, 0)
zero_strain = (axis & strain(zero_motion, gradients, mesh.faces)(axis))
save_figure(render.draw_device(mesh, vertices, areas, zero_motion, zero_potential, zero_strain,
                              displacement_scale, applied_voltage, strain_scale), "piezoelectric_setup")


## Constitutive coupling

Two orthogonal material planes specify the directions across and along the poling. Their extension and shear readings build the elastic work form and the polarization map; turning the planes turns the material response. The electric field pairs with polarization as `electric & (polarized & time)`. At controlled voltage, electric enthalpy includes work exchanged with the source, giving its electric term a negative sign.

The PZT-5A constants below are reduced to plane stress from [Martins and Cordioli, Table A3](https://dael.euracoustics.org/bin/EAA/aaua_dl?document_id=52105). The two entries in each array follow the two material planes; permittivity is measured at fixed in-plane strain.

In [ ]:
directions = stack((mv.x, mv.y))

# Entries follow the two material directions: transverse and parallel to poling.
stretch_stiffness = np.array([73.38258, 64.01459]) * 1e9 / stress_unit
cross_stiffness = 28.18203 * 1e9 / stress_unit
shear_stiffness = 21.05263 * 1e9 / stress_unit
stretch_polarization = np.array([-2.008341, 19.12233]) * field_unit / stress_unit
shear_polarization = 12.29474 * field_unit / stress_unit
permittivities = np.array([8.137619, 7.556942]) * 1e-9 / permittivity_unit

In [ ]:
def constitutive(directions: Plane, stretch_stiffness: np.ndarray, cross_stiffness: float,
                 shear_stiffness: float, stretch_polarization: np.ndarray,
                 shear_polarization: float, permittivities: np.ndarray
                 ) -> tuple[Elastic, Polarization, Permittivity]:
    """Constitutive extensors from orthonormal material planes, transverse then parallel to poling.

    A displacement and its gradient direction describe a change of shape. The elastic form
    pairs two such changes; the piezoelectric map sends one to a polarization direction.
    """
    transverse, poled = directions[..., 0], directions[..., 1]
    # Each material plane reads extension along its own direction, the shape gradient read by the plane's direction;
    # the mixed reading is shear.
    stretches = (directions & Point) * (Plane & directions.dual())
    shear = (transverse & Point) * (Plane & poled.dual()) + (poled & Point) * (Plane & transverse.dual())
    elastic = ((stretches * stretches * stretch_stiffness).sum(axis=-1)
               + cross_stiffness * (stretches[..., 0] * stretches[..., 1]
                                    + stretches[..., 1] * stretches[..., 0])
               + shear_stiffness * shear * shear)
    # Extension polarizes along the poling direction; shear polarizes across it.
    polarization = (poled.dual() * (stretches * stretch_polarization).sum(axis=-1)
                    + transverse.dual() * shear_polarization * shear)
    # The electric field is read on each material direction swept through time.
    permittivity = (directions.dual() * (Field & (directions.dual() & time)) * permittivities).sum(axis=-1)
    return elastic, polarization, permittivity


elastic, polarization, permittivity = constitutive(
    directions, stretch_stiffness, cross_stiffness, shear_stiffness,
    stretch_polarization, shear_polarization, permittivities)

## Coupled equilibrium

A triangle's shape functions read nodal motion as constant strain and nodal potential as electric field. Their adjugates bring stresses and polarization back to nodal forces and charge. The mixed block is built once; `coupling.adjugate()` supplies its reciprocal action. The two diagonal blocks carry elastic stiffness and electric response.

In [ ]:
def assemble(mesh: Mesh, areas: Scalar, gradients: Plane, elastic: Elastic, polarization: Polarization,
             permittivity: Permittivity, poling: Scalar, width: float
             ) -> tuple[SparseExtensor, SparseExtensor, SparseExtensor]:
    """The three blocks of electric enthalpy, as maps to the work-duals of the nodal fields."""
    # Binding each shape gradient turns the continuum laws into nodal work and polarization maps.
    work = elastic(Point, gradients[:, None, :], Point, gradients[:, :, None])
    polarized = polarization(Point, gradients) * poling[:, None]
    electric = -(gradients ^ mv.t) * Scalar

    # Read the work form as the force paired with a test displacement.
    elastic_cells = (Plane & Point).solve(work)
    # Polarization sweeps a plane through time, the electric field's work-dual.
    # Reading this map backwards supplies the converse effect from the same coupling.
    mixed_cells = -(polarized[:, :, None] & time).adjugate()(electric[:, None, :])
    electric_cells = -electric[:, :, None].adjugate()(permittivity(electric[:, None, :]) & time)

    # Local test/trial pairs become couplings between global nodes at assembly.
    weights = width * areas[..., None, None]
    return tuple(SparseExtensor.from_indices(cells * weights, mesh.faces[:, :, None], mesh.faces[:, None, :],
                                             (mesh.n_nodes, mesh.n_nodes))
                 for cells in (elastic_cells, mixed_cells, electric_cells))


mechanical, coupling, dielectric = assemble(mesh, areas, gradients, elastic, polarization, permittivity, poling, width)

## Voltage actuation

A 100 V potential difference drives the strip with no mechanical tip load. Both electrodes are equipotential, while the exposed ends carry no free charge. Orange and blue inside the strip show axial extension and compression; the heavy edges show the signed electrode voltage. The animation sweeps the applied voltage slowly through both signs. Displacements are magnified 100 times.

In [ ]:
def equilibrium(mechanical: SparseExtensor, coupling: SparseExtensor, dielectric: SparseExtensor,
                movement: SparseExtensor, voltage: SparseExtensor,
                prescribed: Scalar, force: Plane) -> tuple[Point, Scalar]:
    """Equilibrium with grounded or prescribed electrodes and zero free charge on floating ones."""
    # The same mixed map supplies forces from voltage and, read backwards, charge from motion.
    elastic = movement.adjugate()(mechanical(movement))
    mixed = movement.adjugate()(coupling(voltage))
    capacitance = voltage.adjugate()(dielectric(voltage))
    load = movement.adjugate()(force - coupling(prescribed))
    charge = -voltage.adjugate()(dielectric(prescribed))
    displacement, potential = solve_coupled(elastic, mixed, capacitance, load, charge)
    return movement(displacement), prescribed + voltage(potential)


movement, driven_voltage = boundary_maps(mesh, np.array([], dtype=int))
actuator_motion, actuator_potential = equilibrium(
    mechanical, coupling, dielectric, movement, driven_voltage,
    electrode_potential(mesh, applied_voltage), tip_load(mesh, 0))
actuator_strain = axis & strain(actuator_motion, gradients, mesh.faces)(axis)
save_figure(render.draw_device(mesh, vertices, areas, actuator_motion, actuator_potential, actuator_strain,
                              displacement_scale, applied_voltage, strain_scale), "piezoelectric_actuator")

In [ ]:
from collections.abc import Iterator


def cycle(displacement: Point, potential: Scalar, axial_strain: Scalar,
          n_frames: int) -> Iterator[tuple[Point, Scalar, Scalar]]:
    """A slow sinusoidal load sweep through equilibria of this linear material."""
    for phase in np.linspace(0, 2 * np.pi, n_frames, endpoint=False):
        amplitude = np.sin(phase)
        yield displacement * amplitude, potential * amplitude, axial_strain * amplitude


show_animation(render.animate(mesh, vertices, areas, cycle(actuator_motion, actuator_potential, actuator_strain, n_frames),
                              displacement_scale, applied_voltage, strain_scale),
               "piezoelectric_actuator", duration_ms, n_frames)

In [ ]:
tip = SparseExtensor.selection(ctx, mesh.tip, mesh.n_nodes)
tip_weights = np.ones(len(mesh.tip))
tip_weights[[0, -1]] = 0.5
actuator_tip = ((mv.y & (tip * actuator_motion)) * mv.scalar(tip_weights[:, None]).field()).sites.sum() / tip_weights.sum()
tip_micrometres = actuator_tip * length_unit * 1e6
print(f"Tip displacement at 100 V: {tip_micrometres.to_array():.2f} μm")

## Open-circuit sensing

A transverse load of 60 mN is distributed over the tip. The bottom electrode is grounded; the top electrode floats with zero total charge. Its common voltage becomes an unknown of the coupled solve. The strip's polarization generates a voltage, and the resulting electric field acts back on its deformation. The electrode colours here span ±10 V between the terminals.

In [ ]:
sensor_voltage_scale = 10 / voltage_unit
movement, floating_voltage = boundary_maps(mesh, mesh.top)
sensor_motion, sensor_potential = equilibrium(
    mechanical, coupling, dielectric, movement, floating_voltage,
    electrode_potential(mesh, 0), tip_load(mesh, applied_force))
sensor_strain = axis & strain(sensor_motion, gradients, mesh.faces)(axis)
save_figure(render.draw_device(mesh, vertices, areas, sensor_motion, sensor_potential, sensor_strain,
                              displacement_scale, sensor_voltage_scale, strain_scale), "piezoelectric_sensor")

In [ ]:
show_animation(render.animate(mesh, vertices, areas, cycle(sensor_motion, sensor_potential, sensor_strain, n_frames),
                              displacement_scale, sensor_voltage_scale, strain_scale),
               "piezoelectric_sensor", duration_ms, n_frames)

In [ ]:
top = SparseExtensor.selection(ctx, mesh.top, mesh.n_nodes)
sensor_volts = (top * sensor_potential).sites.mean() * voltage_unit
print(f"Open-circuit voltage at 60 mN: {sensor_volts.to_array():.2f} V")

## Reciprocal terminal response

Shorting the electrodes lets bending move charge through the circuit. The charge per tip force equals the actuator's tip displacement per volt: both are readings of the same coupling through their work pairings. The actuator also gives the free device's capacitance. Together these predict the open-circuit voltage, where the voltage-induced charge cancels the load-induced charge.

In [ ]:
short_motion, short_potential = equilibrium(
    mechanical, coupling, dielectric, movement, driven_voltage,
    electrode_potential(mesh, 0), tip_load(mesh, applied_force))
# Free charge supplied by the top electrode, the negative potential derivative of enthalpy.
actuator_charge = -(mv.scalar([1]) & (top * (coupling.adjugate()(actuator_motion) + dielectric(actuator_potential)))).sites.sum()
short_charge = -(mv.scalar([1]) & (top * (coupling.adjugate()(short_motion) + dielectric(short_potential)))).sites.sum()
capacitance = actuator_charge / applied_voltage
actuator_gain = actuator_tip / applied_voltage
sensor_gain = short_charge / applied_force
predicted_voltage = -short_charge / capacitance

print(f"Actuator sensitivity: {float((actuator_gain * length_unit / voltage_unit * 1e9).to_array()):.2f} nm/V")
print(f"Charge sensitivity:   {float((sensor_gain * charge_unit / force_unit * 1e9).to_array()):.2f} nC/N")
print(f"Free capacitance:     {float((capacitance * charge_unit / voltage_unit * 1e9).to_array()):.2f} nF")
print(f"Predicted open voltage: {float((predicted_voltage * voltage_unit).to_array()):.2f} V")

The electrical boundary condition changes the mechanical response because the induced field feeds back through the same material coupling. Linear elasticity, dielectric response and piezoelectricity remain separate constitutive maps, joined by their energy pairings.

In stress-charge tensor notation the constitutive law reads $\sigma=C^E\varepsilon-e^\mathsf{T}E$ and $D=e\varepsilon+\kappa^S E$. [ANSYS's bimorph example](https://ansyshelp.ansys.com/public/views/secured/corp/v242/en/ans_cou/Hlp_G_COU3_piezo.html) describes the corresponding actuator and sensor experiments.